# Supplementary Simulation Figure S3 — temporal recovery

Reproduces the finalized temporal-recovery supplement.

Panel A shows temporal recovery across the PCA-favorable, AA-favorable, and
metastable regimes. Panel B shows the paired MS-AA − PCA temporal-recovery
difference in the metastable regime.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_CSV = Path(
    "simulation_pca_msaa_extended_outputs/"
    "pca_msaa_extended_results_paper.csv"
)

OUTPUT_DIR = Path("paper_figure_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not RESULTS_CSV.exists():
    raise FileNotFoundError(
        f"Missing {RESULTS_CSV}. Run revision_16_pca_msaa_clean_simulation_benchmark "
        "with RUN_MODE='paper' first."
    )

results = pd.read_csv(RESULTS_CSV)

METHOD_COLORS = {
    "MS-AA": "#1f77b4",
    "PCA": "#ff7f0e",
}

def mean_sem(df, group_col, value_col):
    return (
        df.groupby(group_col)[value_col]
        .agg(["mean", "sem"])
        .reset_index()
    )

FIG_PNG = OUTPUT_DIR / "supp_simulation_temporal.png"
FIG_PDF = OUTPUT_DIR / "supp_simulation_temporal.pdf"

In [ ]:
core_order = ["PCA_control", "AA_control", "metastable"]
core_labels = ["PCA-favorable", "AA-favorable", "Metastable"]

core = results[
    results["regime"].isin(core_order)
].copy()

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10.5, 4.6),
    gridspec_kw={"wspace": 0.30},
)

axA, axB = axes

x = np.arange(len(core_order))
offset = 0.12

for method, col, shift in [
    ("MS-AA", "aa_temporal_recovery", -offset),
    ("PCA", "pca_temporal_recovery", offset),
]:
    means = []
    sems = []

    for regime in core_order:
        vals = core.loc[
            core["regime"].astype(str) == regime,
            col,
        ]
        means.append(vals.mean())
        sems.append(vals.sem())

    axA.errorbar(
        x + shift,
        means,
        yerr=np.asarray(sems) * 1.96,
        marker="o",
        linewidth=1.8,
        capsize=3,
        color=METHOD_COLORS[method],
        label=method,
    )

axA.set_xticks(x)
axA.set_xticklabels(core_labels, rotation=12, ha="right")
axA.set_ylim(0, 1.05)
axA.set_ylabel("Temporal recovery (r)")
axA.set_title("Temporal recovery across calibration regimes", fontweight="bold")
axA.legend(frameon=False)

meta = core[
    core["regime"].astype(str) == "metastable"
].copy()

delta = (
    meta["aa_temporal_recovery"].to_numpy(dtype=float)
    - meta["pca_temporal_recovery"].to_numpy(dtype=float)
)

rng = np.random.default_rng(20260908)
boot = np.array([
    rng.choice(delta, size=len(delta), replace=True).mean()
    for _ in range(20000)
])

mean_delta = float(np.mean(delta))
lo, hi = np.quantile(boot, [0.025, 0.975])

jitter = np.linspace(-0.035, 0.035, len(delta))
axB.scatter(
    jitter,
    delta,
    s=24,
    alpha=0.35,
    color=METHOD_COLORS["MS-AA"],
)

axB.errorbar(
    [0],
    [mean_delta],
    yerr=np.array([
        [mean_delta - lo],
        [hi - mean_delta],
    ]),
    fmt="o",
    markersize=8,
    capsize=5,
    linewidth=2,
    color=METHOD_COLORS["MS-AA"],
)

axB.axhline(0, color="0.55", linestyle="--", linewidth=1)
axB.set_xticks([0])
axB.set_xticklabels(["MS-AA − PCA"])
axB.set_ylabel("Paired temporal-recovery difference (Δr)")
axB.set_title("Metastable paired temporal recovery", fontweight="bold")

for ax, letter in zip(axes, ["A", "B"]):
    ax.text(
        -0.10, 1.04, letter,
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(
    "Temporal recovery across calibration controls and the metastable regime",
    fontsize=15,
    fontweight="bold",
)

fig.savefig(FIG_PNG, dpi=300, bbox_inches="tight")
fig.savefig(FIG_PDF, bbox_inches="tight")
plt.show()

print("Saved:", FIG_PNG)
print("Saved:", FIG_PDF)